In [1]:
from unittest import skip

import pandas as pd
import numpy as np

# Daten einlesen

In [2]:
df = pd.read_csv("../data/processed/swissvotes_processed.csv")
df.head()

,anr,datum,legisjahr,jahrzehnt,rechtsform_name,titel_kurz_d,anzahl,beteiligung,annahme,volkja-proz,...,sg-japroz,gr-japroz,ag-japroz,tg-japroz,ti-japroz,vd-japroz,vs-japroz,ne-japroz,ge-japroz,ju-japroz
0,1.0,1848-09-12,1848-1851,1840,Obligatorisches Referendum,Bundesverfassung der schweizerischen Eidgenoss...,1,NaN,1.0,72.83,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,1866-01-14,1863-1866,1860,Obligatorisches Referendum,Mass und Gewicht,9,NaN,0.0,50.44,...,26.58,5.62,46.78,84.56,88.62,66.56,48.18,89.41,87.64,NaN
2,3.0,1866-01-14,1863-1866,1860,Obligatorisches Referendum,Gleichstellung der Juden und Naturalisierten m...,9,NaN,1.0,53.23,...,29.87,10.35,63.32,85.27,84.51,62.70,37.89,85.41,85.95,NaN
3,4.0,1866-01-14,1863-1866,1860,Obligatorisches Referendum,Stimmrecht der Niedergelassenen in Gemeindeang...,9,NaN,0.0,43.08,...,27.99,9.89,60.67,84.98,63.39,10.65,15.91,71.95,37.97,NaN
4,5.0,1866-01-14,1863-1866,1860,Obligatorisches Referendum,Besteuerung und zivilrechtliche Verhältnisse d...,9,NaN,0.0,39.88,...,24.29,10.45,50.71,81.12,59.32,6.79,13.13,74.70,58.28,NaN


## Kongruenz ohne Mobilisierung (Framing nicht mehr Stimmberechtigte sondern in Bezug auf

In [ ]:
df_with_positions = df.copy()
neue_spalten = {}

for col in df:
    scores = []
    for i, row in df_with_positions.iterrows():
        position = row[col]
        ja_proz = row['volkja-proz']
        if col.startswith('p-') or col.endswith('-pos'):

            # Ja-Parole oder Volksinitiative bevorzugt
            if position in [1.0, 9.0]:
                scores.append((ja_proz - 50) / 100)

            # Nein-Parole oder Gegenentwurf bevorzugt
            elif position in [2.0, 8.0]:
                scores.append((50 - ja_proz) / 100)

            # Neutral
            elif position in [3.0, 4.0, 5.0, 66.0]:
                scores.append(0.0)

            else:
                scores.append(np.nan)

    # Partei- und Positions-Spalten speichern
    if col.startswith('p-') or col.endswith('-pos'):
        neue_spalten[f'zustimmung_{col}'] = scores

df_with_positions = pd.concat(
    [df_with_positions, pd.DataFrame(neue_spalten, index=df_with_positions.index)],
    axis=1
)

df_with_positions.head()

In [13]:
df_with_positions.to_csv('../data/processed/df_with_positions.csv', index=False)

### Datenset für Heatmap

In [6]:
df_pos = df.copy() # Kopie des Datensets
parteien_cols = [c for c in df_pos.columns if str(c).startswith("p-")] # Liste der Partei-Spalten
position_cols = ['br-pos','bv-pos']+parteien_cols # verschiedene Positionen zusammenhängen.

canton_cols = [c for c in df.columns if str(c).endswith('-japroz')] # Liste der Kanton-Spalten

rows = [] #leere Liste für die Zeilen

# Dreistufige Iteration über die Zeilen, die Positions-Spalten (Parteien, br-pos, bv-pos) und die Kantone
for idx, row in df_pos.iterrows(): # 1. Iteration Zeile (Abstimmung)
    for partei in position_cols: # 2. Iteration Positions-Spalte
        zeile = {
            "abstimmung": idx, #Spalte Abstimmung
            "partei": partei, #Spalte Partei / Institution
        }
        for kanton in canton_cols: # 3. Iteration Kanton
            ja_proz = row[kanton]
            position = row[partei]
            # Ja-Parole oder Volksinitiative bevorzugt
            if position in [1.0, 9.0]:
                zeile[kanton] = (ja_proz - 50)
            # Nein-Parole oder Gegenentwurf bevorzugt
            elif position in [2.0, 8.0]:
               zeile[kanton] = (50 - ja_proz)
            # Neutral
            elif position in [3.0, 4.0, 5.0, 66.0]:
                zeile[kanton] = 0.0

            else:
                zeile[kanton] = np.nan
        rows.append(zeile)

df_heatmap = pd.DataFrame(rows)
df_heatmap_with_positions = df_heatmap.drop(columns=["abstimmung"]).groupby("partei").mean(numeric_only=True) # Gruppieren nach Partei und Berechnen des Mittelwerts.
df_heatmap_with_positions.head()

,zh-japroz,be-japroz,lu-japroz,ur-japroz,sz-japroz,ow-japroz,nw-japroz,gl-japroz,zg-japroz,fr-japroz,...,sg-japroz,gr-japroz,ag-japroz,tg-japroz,ti-japroz,vd-japroz,vs-japroz,ne-japroz,ge-japroz,ju-japroz
partei,,,,,,,,,,,,,,,,,,,,,
br-pos,10.826331,10.065737,10.597266,7.399514,6.714335,9.188903,10.239083,9.566313,11.069550,9.562248,...,9.605504,11.319496,9.629676,10.933867,8.050108,10.613399,7.687248,9.237212,9.360629,7.575995
bv-pos,12.113401,10.353255,10.441635,6.855226,5.759197,8.913635,9.383766,9.826350,10.747898,9.469358,...,9.354496,10.792409,9.186365,11.262555,10.040161,10.737708,7.490131,10.038905,10.958920,7.645403
p-acs,3.945455,5.906364,9.110000,8.553636,12.110909,13.237273,12.630000,9.477273,7.893636,10.120909,...,8.840000,-0.444545,10.727273,9.321818,6.649091,7.773636,10.901818,10.496364,5.812727,4.315000
p-bdp,11.664571,10.562381,12.494190,9.952000,10.047619,11.852000,12.737619,9.650667,13.748571,11.436381,...,10.123333,12.478286,10.823048,9.836381,8.432667,13.588000,11.948952,11.060000,10.401429,8.860381
p-bpuk,21.260000,16.830000,18.070000,5.930000,6.640000,6.290000,9.120000,16.590000,21.410000,12.870000,...,14.340000,11.480000,16.880000,18.570000,5.320000,6.450000,-30.350000,17.670000,7.750000,12.800000


In [7]:
df_heatmap_with_positions.to_csv('../data/processed/df_heatmap_with_positions.csv', index=True)